<a href="https://colab.research.google.com/github/mahashabdekedar/Quantum_Computing_Projects/blob/main/MultilayerQNNStripesClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [113]:
# Setup
!pip install numpy
!pip install scipy
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit_ibm_runtime

We take black and white 3x3 images representing stripes (horizontal) or bars (vertical), and attempt to classify them using a quantum neural network. We choose our network to be a set of rotations, followed by a layer of CZ gates, followed by another group of rotations, and then measurement. We flatten images into arrays of length 9, where array[0] represents the top left qubit and array[2] represents the top right. Let $\vert 1\rangle$ be the correct label for a stripe, and $\vert 0\rangle$ be the correct label for a bar.

In [114]:
# Generate a training set
import numpy as np

# Stripes
stripes = []

for i in range(3):
  blank = np.ones((3, 3), np.int8)
  blank[i,:] = -1
  stripes.append(blank.flatten())
stripes.extend([mat * -1 for mat in stripes])

# Bars
bars = []

for i in range(3):
  blank = np.ones((3, 3), np.int8)
  blank[:,i] = -1
  bars.append(blank.flatten())

bars.extend([mat * -1 for mat in bars])

features = bars + stripes

# labels
labels  = np.concatenate((np.ones(len(bars)), -np.ones(len(stripes))))


We'll add some noise to our dataset, and holdout some examples for validation

In [115]:
import random

rng = random.Random(42)
np_rng = np.random.default_rng(42)

# Let's add some noise to our data

noisy_features = []
noisy_labels = []
for feat, lab in zip(features, labels):
  for i in range(3):
    noisy_features.append(feat + np_rng.normal(loc=0, scale = 0.1, size=9))
    noisy_labels.append(lab)

# Holdout roughly 20%
holdout = set(rng.sample(range(0, len(noisy_features)), 8))
use = set(range(len(noisy_features))).difference(holdout)


x_train = [noisy_features[x] for x in use]
x_validate = [noisy_features[x] for x in holdout]

y_train = [noisy_labels[x] for x in use]
y_validate = [noisy_labels[x] for x in holdout]



# rng = random.Random(42)
# while len(holdout) != 2:
#   holdout.add(rng.randint(0, 11))

# use = set(range(12)).difference(holdout)

# x_train = [features[x] for x in use]
# y_train = [labels[x] for x in use]

# x_validate = [features[x] for x in holdout]
# y_validate = [labels[x] for x in holdout]

# print(x_validate)
# print(y_validate)



Now for the actual circuit. We'll setup and transpile a parameterized version, and assign values to the parameters later. Note that we choose the order of rotations y, z, x because if z were first or last it would have no effect either on the input basis state or on the measurement outcome of the output state. The order of y and x is arbitrary. Furthermore we choose CZ over CX because they commute.

We'll attach an encoder to pass the correct input state. We choose to encode input states using an x rotation on every qubit, with magnitude corresponding the value of the matching pixel. We map $[-1, 1]$ onto $[0, \pi]$.

In [116]:
from qiskit.circuit import ParameterVector, QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import time



def set_params(parameters, qc, param_vec_theta):
  return qc.assign_parameters(dict(zip(param_vec_theta, parameters)))

def set_encoding(arr, qc, param_vec_phi):
  return qc.assign_parameters(dict(zip(param_vec_phi, arr)))

# Build the circuit
def buildCircuit(num_layers=1):
  qc = QuantumCircuit(9)
  phi = ParameterVector("phi", 9)
  theta = ParameterVector("theta", 27*num_layers + 3)

  # Encoding Layer
  for i in range(9):
    qc.rx(phi[i], i)

  for i in range(num_layers):
    # Add a layer of rotations
    for j in range(9):
      qc.ry(theta[27*i + 3*j], j)
      qc.rz(theta[27*i + 3*j + 1], j)
      qc.rx(theta[27*i + 3*j + 2], j)

    # Add an entanglement layer
    qc.cz(0, 1)
    qc.cz(1, 2)
    qc.cz(3, 4)
    qc.cz(4, 5)
    qc.cz(6, 7)
    qc.cz(7, 8)
    qc.cz(0, 3)
    qc.cz(1, 4)
    qc.cz(2, 5)
    qc.cz(3, 6)
    qc.cz(4, 7)
    qc.cz(5, 8)

  # Add last layer of rotations
  qc.ry(theta[27*num_layers], 4)
  qc.rz(theta[27*num_layers + 1], 4)
  qc.rx(theta[27*num_layers + 2], 4)


  return qc, phi, theta
main, phi_main, theta_main = buildCircuit(3)
main.draw()

┌────────────┐ ┌──────────────┐ ┌──────────────┐ ┌──────────────┐         »
q_0: ┤ Rx(phi[0]) ├─┤ Ry(theta[0]) ├─┤ Rz(theta[1]) ├─┤ Rx(theta[2]) ├─■─────■─»
     ├────────────┤ ├──────────────┤ ├──────────────┤ ├──────────────┤ │     │ »
q_1: ┤ Rx(phi[1]) ├─┤ Ry(theta[3]) ├─┤ Rz(theta[4]) ├─┤ Rx(theta[5]) ├─■──■──┼─»
     ├────────────┤ ├──────────────┤ ├──────────────┤ ├──────────────┤    │  │ »
q_2: ┤ Rx(phi[2]) ├─┤ Ry(theta[6]) ├─┤ Rz(theta[7]) ├─┤ Rx(theta[8]) ├────■──┼─»
     ├────────────┤ ├──────────────┤┌┴──────────────┤┌┴──────────────┤       │ »
q_3: ┤ Rx(phi[3]) ├─┤ Ry(theta[9]) ├┤ Rz(theta[10]) ├┤ Rx(theta[11]) ├─■─────■─»
     ├────────────┤┌┴──────────────┤├───────────────┤├───────────────┤ │       »
q_4: ┤ Rx(phi[4]) ├┤ Ry(theta[12]) ├┤ Rz(theta[13]) ├┤ Rx(theta[14]) ├─■──■────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤    │    »
q_5: ┤ Rx(phi[5]) ├┤ Ry(theta[15]) ├┤ Rz(theta[16]) ├┤ Rx(theta[17]) ├────■────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤         »
q_6: ┤ Rx(phi[6]) ├┤ Ry(theta[18]) ├┤ Rz(theta[19]) ├┤ Rx(theta[20]) ├─■───────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤ │       »
q_7: ┤ Rx(phi[7]) ├┤ Ry(theta[21]) ├┤ Rz(theta[22]) ├┤ Rx(theta[23]) ├─■──■────»
     ├────────────┤├───────────────┤├───────────────┤├───────────────┤    │    »
q_8: ┤ Rx(phi[8]) ├┤ Ry(theta[24]) ├┤ Rz(theta[25]) ├┤ Rx(theta[26]) ├────■────»
     └────────────┘└───────────────┘└───────────────┘└───────────────┘         »
«     ┌───────────────┐┌───────────────┐┌───────────────┐                 »
«q_0: ┤ Ry(theta[27]) ├┤ Rz(theta[28]) ├┤ Rx(theta[29]) ├─────────────────»
«     └───────────────┘├───────────────┤├───────────────┤┌───────────────┐»
«q_1: ────────■────────┤ Ry(theta[30]) ├┤ Rz(theta[31]) ├┤ Rx(theta[32]) ├»
«             │        └───────────────┘├───────────────┤├───────────────┤»
«q_2: ────────┼────────────────■────────┤ Ry(theta[33]) ├┤ Rz(theta[34]) ├»
«             │                │        └───────────────┘├───────────────┤»
«q_3: ────────┼────────────────┼────────────────■────────┤ Ry(theta[36]) ├»
«             │                │                │        └───────────────┘»
«q_4: ────────■────────────────┼────────────────┼────────────────■────────»
«                              │                │                │        »
«q_5: ─────────────────────────■────────────────┼────────────────┼────────»
«                                               │                │        »
«q_6: ──────────────────────────────────────────■────────────────┼────────»
«                                                                │        »
«q_7: ───────────────────────────────────────────────────────────■────────»
«                                                                         »
«q_8: ────────────────────────────────────────────────────────────────────»
«                                                                         »
«                                                                            »
«q_0: ────────■────────────────────────────────────────────────────────────■─»
«             │                                                            │ »
«q_1: ────────■────────────────■───────────────────────────────────────────┼─»
«     ┌───────────────┐        │                                           │ »
«q_2: ┤ Rx(theta[35]) ├────────■───────────────────────────────────────────┼─»
«     ├───────────────┤┌───────────────┐                                   │ »
«q_3: ┤ Rz(theta[37]) ├┤ Rx(theta[38]) ├─────────────────────────■─────────■─»
«     ├───────────────┤├───────────────┤┌───────────────┐        │           »
«q_4: ┤ Ry(theta[39]) ├┤ Rz(theta[40]) ├┤ Rx(theta[41]) ├────────■─────────■─»
«     └───────────────┘├───────────────┤├───────────────┤┌───────────────┐ │ »
«q_5: ────────■────────┤ Ry(theta[42]) ├┤ Rz(theta[43]) ├┤ Rx(theta[44]) ├─■─»
«             │        ├───────────────┤├───────────────┤├───────────────┤

Now we create a function that will take in parameters and determine the loss, so we can simply pass this to scipy.optimize.minimize and have the optimization done for us. We choose to use mean square error as our loss function for simplicity here.

In [117]:
from qiskit_aer.primitives import EstimatorV2
import time

def mse(predictions, labels):
  diff = np.array(predictions) - labels
  return (diff @ diff) / len(diff)

estimator = EstimatorV2()


# def get_expectations(qc : QuantumCircuit, features, param_vec_phi: ParameterVector, num_shots=1000):
#   expectations = []

#   for feat in features:
#     qc_bound = set_encoding(np.pi * (feat + 1)/2, qc, param_vec_phi)

#     result = estimator.run([qc_bound]).result()

#     counts = result[0].data.c.get_counts()

#     p1 = counts.get("1", 0) / num_shots
#     expectations.append(1-2*p1)

#   return expectations

# print(get_expectations(circuits, x_train))

def get_expectations(parameters, qc: QuantumCircuit, features):


  mat_params = np.hstack([np.pi * (np.array(features)+ 1) / 2, np.tile(parameters, (len(features), 1))])

  job = estimator.run([(qc, "IIIIZIIII", mat_params)])
  result = job.result()

  return result[0].data.evs

def get_loss(parameters, qc: QuantumCircuit, features, labels):


  mat_params = np.hstack([np.pi * (np.array(features)+ 1) / 2, np.tile(parameters, (len(features), 1))])

  job = estimator.run([(qc, "IIIIZIIII", mat_params)])
  result = job.result()

  expectations = result[0].data.evs


  # Return loss
  return mse(expectations, labels)
start = time.perf_counter()
print(get_loss(np.zeros(len(theta_main)), main, x_train, y_train))
end = time.perf_counter()
print(end - start)




1.96794869526872
0.15573682400008693


Now all that's left is to run the optimizer!

In [118]:
from scipy.optimize import minimize



rng = np.random.default_rng(42)
ansatz = rng.random(len(theta_main))


start_time = time.perf_counter()
angles  = minimize(get_loss, ansatz, args=(main, x_train, y_train), method="COBYLA", options={"maxiter":1000})

end_time = time.perf_counter()
print(f"Training time: {end_time-start_time} seconds")

Training time: 70.49587602999964 seconds


In [119]:
print(angles)

print(f"weights: {angles.x}")

# print(f"loss: {get_loss(angles.x, theta_main, circuits, x_train, y_train)}")

 message: Return from COBYLA because the objective function has been evaluated MAXFUN times.
 success: False
  status: 3
     fun: 0.08661360642848394
       x: [ 3.511e-01  1.564e+00 ...  8.786e-01  1.241e+00]
    nfev: 1000
   maxcv: 0.0
weights: [ 3.51140937e-01  1.56368642e+00  3.68544982e-01  8.93483230e-01
 -9.87761443e-01  2.28382073e+00  3.37485451e-02  7.24857384e-01
  2.28228229e-03  8.96777307e-01  4.64426332e-01  1.62228393e+00
  1.38802931e+00  6.60302793e-01 -3.17129764e-01  1.17898591e+00
  2.46582007e+00  6.15904233e-01  2.10751762e+00  1.95644484e+00
  6.49176966e-01  9.49418835e-01  1.46861251e+00  9.65732766e-01
  1.20667422e-01  2.08523712e+00  2.28195475e-01 -9.64955559e-02
  2.33288098e+00 -2.09046113e-02  1.87547292e+00  1.41465770e+00
  2.54953277e-01  7.60151304e-01  1.52223523e+00  7.61670094e-01
  2.08813527e-01  1.58267547e+00  1.87830853e-01  1.47919080e+00
 -4.79552675e-01  1.01654369e+00  2.51235780e-01  1.55426163e+00
  2.54376736e-01  8.56373710e-01  2.

We can also check if it classifies correctly.

In [120]:
optimal_circuit = set_params(angles.x, main, theta_main)

output = get_expectations(angles.x, main, x_train)
for y_hat, y in zip(output, y_train):
  if abs(y_hat) < 0: # Deprecated
    print(f"Low confidence. Actual: {y}")
  elif y_hat * y >= 0:
    print("Correct!")
  else:
    print(f"Wrong :(. Guess: {y_hat}. Actual: {y}")

Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!


Now out of sample:

In [121]:
validation_expectations = get_expectations(angles.x, main, x_validate)
for y_hat, y in zip(validation_expectations, y_validate):
  if abs(y_hat) < 0: # Deprecated
    print(f"Low confidence. Actual: {y}")
  elif y_hat * y >= 0:
    print("Correct!")
  else:
    print(f"Wrong :(. Guess: {y_hat}. Actual: {y}")

Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
Correct!
